# 02 — Health Speech Identification

This notebook constructs a topic-based selection of health-related Italian parliamentary speeches.

BERTopic groups speeches using semantic embeddings computed from cleaned texts. Lemmatized texts provide the keywords used to interpret the topics.

The main selection includes the general health topic. An extended selection also includes the topic concerning end-of-life care and related ethical debates.

Topic choices were informed by exploratory inspection of topic keywords, sampled speeches and unassigned records. These checks identified both off-topic inclusions and relevant speeches outside the selected topics. The resulting selections are approximate.

All eligible party categories are retained. The five-party restriction is applied in the subsequent linguistic analysis.

Run the cells in order in Google Colab with a GPU runtime. Files under `/content` are temporary; download the results before ending the session.

**Input:** `data/processed/corpus_prepared.parquet`

**Main output:** `02_health_identification/outputs/corpus_health_bertopic.parquet`

## 1. Load the prepared corpus

Download the prepared corpus from the project repository and select records eligible for topic modelling. Reuse the local file if it is already available.

In [17]:
from pathlib import Path

import pandas as pd
import requests

PROJECT_DIR = Path("/content/italian-parliament-health-discourse")
CORPUS_PATH = PROJECT_DIR / "data/processed/corpus_prepared.parquet"

CORPUS_URL = (
    "https://media.githubusercontent.com/media/"
    "MnGianmaria/Italian-parliament-health-discourse/main/"
    "data/processed/corpus_prepared.parquet"
)

CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CORPUS_PATH.exists():
    temporary_path = CORPUS_PATH.with_suffix(".parquet.part")

    with requests.get(
        CORPUS_URL, stream=True, timeout=(30, 120)
    ) as response:
        response.raise_for_status()

        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)

    temporary_path.replace(CORPUS_PATH)

corpus = pd.read_parquet(CORPUS_PATH)

eligible = (
    corpus.loc[corpus["eligible_for_models"]]
    .copy()
    .reset_index(drop=True)
)

if eligible["row_id"].isna().any() or eligible["row_id"].duplicated().any():
    raise ValueError("Eligible speeches must have non-missing, unique IDs.")

print(f"Prepared records: {len(corpus):,}")
print(f"Eligible speeches: {len(eligible):,}")

Prepared records: 61,985
Eligible speeches: 49,629


## 2. Set up semantic embeddings

Load a multilingual Sentence Transformer to encode the cleaned Italian speeches.

Split long speeches into non-overlapping passages within the model's token limit. Combine passage embeddings using a token-weighted mean, then normalize the resulting speech vectors.

This procedure processes the full cleaned text and produces one vector per speech, summarizing its overall semantic content.

In [2]:
%pip install -q sentence-transformers bertopic

In [18]:
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before generating embeddings.")

MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(MODEL_NAME, device="cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_NAME)
print("Maximum sequence length:", embedding_model.max_seq_length)
print("Embedding dimensions:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

GPU: Tesla T4
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Maximum sequence length: 128
Embedding dimensions: 384


## 3. Generate speech embeddings

Split cleaned speeches into non-overlapping passages within the model's token limit, including special tokens.

Compute passage embeddings and combine them into a speech embedding using a token-weighted mean. Normalize the resulting vectors and save them together with the speech identifiers to preserve alignment.

In [4]:
import numpy as np
from tqdm.auto import tqdm

OUTPUT_DIR = PROJECT_DIR / "02_health_identification/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = embedding_model.tokenizer
embedding_model.eval()

speech_embeddings = []
total_passages = 0

# Process small groups of speeches to keep memory use bounded.
for start in tqdm(
    range(0, len(eligible), 128),
    desc="Embedding speeches",
):
    texts = eligible["text_clean"].iloc[start:start + 128].tolist()

    encoded = tokenizer(
        texts,
        max_length=embedding_model.max_seq_length,
        truncation=True,
        stride=0,
        return_overflowing_tokens=True,
        return_special_tokens_mask=True,
        padding=True,
        return_tensors="pt",
    )

    speech_index = encoded.pop("overflow_to_sample_mapping").numpy()
    special_tokens = encoded.pop("special_tokens_mask")

    # Weight each passage by its number of non-special, non-padding tokens.
    weights = (
        encoded["attention_mask"] * (1 - special_tokens)
    ).sum(dim=1).numpy()

    passage_vectors = []

    with torch.inference_mode():
        for offset in range(0, len(speech_index), 64):
            features = {
                key: value[offset:offset + 64].to("cuda")
                for key, value in encoded.items()
            }

            vectors = embedding_model(features)["sentence_embedding"]
            passage_vectors.append(vectors.cpu().numpy())

    passage_vectors = np.concatenate(passage_vectors)

    combined = np.zeros(
        (len(texts), passage_vectors.shape[1]),
        dtype=np.float32,
    )
    np.add.at(
        combined,
        speech_index,
        passage_vectors * weights[:, None],
    )

    token_totals = np.bincount(
        speech_index, weights=weights, minlength=len(texts)
    )
    combined /= token_totals[:, None]
    combined /= np.linalg.norm(combined, axis=1, keepdims=True)

    speech_embeddings.append(combined)
    total_passages += len(speech_index)

embeddings = np.concatenate(speech_embeddings)

if not np.isfinite(embeddings).all():
    raise ValueError("Embedding vectors contain invalid values.")

EMBEDDINGS_PATH = OUTPUT_DIR / "speech_embeddings.npz"

np.savez_compressed(
    EMBEDDINGS_PATH,
    embeddings=embeddings,
    row_ids=eligible["row_id"].to_numpy(dtype=str),
    model_name=np.array(MODEL_NAME),
)

print(f"Passages encoded: {total_passages:,}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Saved: {EMBEDDINGS_PATH.name}")

Embedding speeches:   0%|          | 0/388 [00:00<?, ?it/s]

Passages encoded: 277,977
Embedding shape: (49629, 384)
Saved: speech_embeddings.npz


## 4. Fit BERTopic

Cluster speeches using the precomputed semantic embeddings. Use lemmatized texts to extract topic keywords.

UMAP reduces embedding dimensionality before HDBSCAN clustering. The minimum cluster size is set to 100 speeches; the number of topics is determined by the clustering procedure.

Topic -1 contains speeches not assigned to a cluster. These records remain available for subsequent inspection.

In [21]:
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# Verify that embedding rows match the current speech order.
with np.load(EMBEDDINGS_PATH, allow_pickle=False) as saved:
    if not np.array_equal(
        saved["row_ids"],
        eligible["row_id"].to_numpy(dtype=str),
    ):
        raise ValueError("Embedding IDs do not match the speech order.")

    embeddings = saved["embeddings"]

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

cluster_model = HDBSCAN(
    min_cluster_size=100,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=20_000,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(
    eligible["text_lexical"].tolist(),
    embeddings=embeddings,
)

topic_summary = topic_model.get_topic_info()

assignments = eligible[["row_id"]].copy()
assignments["bertopic_topic"] = topics

assignments.to_parquet(
    OUTPUT_DIR / "bertopic_assignments.parquet",
    index=False,
)

print(f"Topics found: {(topic_summary['Topic'] != -1).sum():,}")
print(f"Unassigned speeches: {assignments['bertopic_topic'].eq(-1).sum():,}")


2026-10-29 18:37:54,748 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-29 18:38:52,133 - BERTopic - Dimensionality - Completed ✓
2026-10-29 18:38:52,136 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-29 18:38:58,661 - BERTopic - Cluster - Completed ✓
2026-10-29 18:38:58,674 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-29 18:39:34,799 - BERTopic - Representation - Completed ✓


Topics found: 69
Unassigned speeches: 20,629


In [20]:
topic_overview = topic_summary[["Topic", "Count"]].copy()

topic_overview["keywords"] = [
    ", ".join(word for word, _ in topic_model.get_topic(topic))
    for topic in topic_overview["Topic"]
]

topic_overview.to_csv(
    OUTPUT_DIR / "bertopic_topics.csv",
    index=False,
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 180,
):
    display(topic_overview)

,Topic,Count,keywords
0,-1,20629,"non, presidente, potere, dovere, volere, collega, legge, italia, chiedere, emendamento"
1,0,3415,"sanitario, non, salute, vaccino, medico, pass, green pass, green, regione, potere"
2,1,1530,"scuola, scolastico, docente, studente, istruzione, insegnante, universit, non, formazione, ricerca"
3,2,1458,"fiscale, bilancio, non, impresa, euro, crescita, tassa, debito, spesa, pubblico"
4,3,1435,"lavoratore, reddito, contratto, azienda, non, cittadinanza, reddito cittadinanza, sociale, impresa, decreto"
5,4,1242,"presidente, aula, non, collega, parlamento, chiedere, ordine, presidenza, signore, signore presidente"
6,5,940,"giustizia, processo, riforma, magistrato, penale, pena, reato, non, prescrizione, giudice"
7,6,922,"europeo, unione, europa, europea, unione europea, italia, presidente, non, politica, membro"
8,7,894,"immigrazione, accoglienza, non, italiano, decreto, clandestino, immigrato, sicurezza, italia, salvini"
9,8,815,"energia, energetico, gas, prezzo, rinnovabile, transizione, climatico, costo, fonte, impresa"


## 5. Select health-related topics

The main selection includes topic 0, concerning healthcare and health policy. The extended selection also includes topic 14, concerning end-of-life care and related ethical debates.

Topic numbers refer to the fitted model reported above. When refitting in a different environment, inspect topic keywords before reusing these numbers.

In [24]:
selection = eligible.merge(
    assignments,
    on="row_id",
    how="left",
    validate="one_to_one",
)

MAIN_HEALTH_TOPICS = [0]
EXTENDED_HEALTH_TOPICS = [0, 14]

selection["health_bertopic_main"] = (
    selection["bertopic_topic"].isin(MAIN_HEALTH_TOPICS)
)
selection["health_bertopic_extended"] = (
    selection["bertopic_topic"].isin(EXTENDED_HEALTH_TOPICS)
)

print(f"Main selection: {selection['health_bertopic_main'].sum():,}")
print(f"Extended selection: {selection['health_bertopic_extended'].sum():,}")

Main selection: 3,415
Extended selection: 3,876


## 6. Export the health selection

Save topic assignments and selection flags for all eligible speeches. Export the main health subset with its original metadata and prepared text representations, retaining all party categories.

The extended subset can be reconstructed by joining the selection flags to the prepared corpus using `row_id`.

In [25]:
selection[
    [
        "row_id",
        "bertopic_topic",
        "health_bertopic_main",
        "health_bertopic_extended",
    ]
].to_parquet(
    OUTPUT_DIR / "bertopic_selection.parquet",
    index=False,
)

health_corpus = selection.loc[
    selection["health_bertopic_main"]
].copy()

health_path = OUTPUT_DIR / "corpus_health_bertopic.parquet"

health_corpus.to_parquet(
    health_path,
    index=False,
    compression="zstd",
)

print(f"Exported health speeches: {len(health_corpus):,}")
print(f"Saved: {health_path.name}")

Exported health speeches: 3,415
Saved: corpus_health_bertopic.parquet


## 7. Download the results (optional)

Package the final outputs and run metadata in one archive. Only the files listed below are included; exploratory files from earlier work in the same session are excluded.

The metadata records software versions and model settings. Exact topic assignments may differ across software environments.

In [ ]:
import json
import zipfile
from importlib.metadata import version

from google.colab import files

run_metadata = {
    "embedding_model": MODEL_NAME,
    "maximum_sequence_length": embedding_model.max_seq_length,
    "embedding_aggregation": "token-weighted mean of non-overlapping passages, L2 normalized",
    "umap": {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0,
             "metric": "cosine", "random_state": 42},
    "hdbscan": {"min_cluster_size": 100, "min_samples": 10,
                "metric": "euclidean", "cluster_selection_method": "eom"},
    "vectorizer": {"ngram_range": [1, 2], "max_features": 20000},
    "main_health_topics": MAIN_HEALTH_TOPICS,
    "extended_health_topics": EXTENDED_HEALTH_TOPICS,
    "eligible_speeches": len(eligible),
    "main_health_speeches": len(health_corpus),
    "versions": {
        package: version(package)
        for package in ["torch", "transformers", "sentence-transformers",
                        "bertopic", "umap-learn", "hdbscan", "scikit-learn",
                        "numpy", "pandas", "pyarrow"]
    },
}

metadata_path = OUTPUT_DIR / "run_metadata.json"
metadata_path.write_text(
    json.dumps(run_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

output_files = [
    "speech_embeddings.npz",
    "bertopic_assignments.parquet",
    "bertopic_topics.csv",
    "bertopic_selection.parquet",
    "corpus_health_bertopic.parquet",
    "run_metadata.json",
]

archive_path = PROJECT_DIR / "health_identification_outputs.zip"

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
    for filename in output_files:
        archive.write(OUTPUT_DIR / filename, arcname=filename)

files.download(str(archive_path))